# IAEA3d

### This is acomputation of the IAEA3d benchmark ![](./IAEA3D.png)

In [1]:
import numpy as np
from numba.np.unsafe.ndarray import *

from dorban.finite_differences.finite_difference_current_calculator import \
    CurrentCalculatorFD
from dorban.finite_differences.solve_finite_difference import solve_k_diffusion
from dorban.geometry.boundary_conditions import (Boundary, Void,
                                                 Reflector)
from dorban.geometry.cartesian import Cartesian
from dorban.geometry.polybox import PolyBox
from dorban.materials import Fissionable, Isotope
from dorban.system import Core

## Defining the materials

In [2]:
mat1 = Fissionable("Fuel1", scatter=np.array([[0, 0], [0.02, 0]]),
                   absorb=np.array([0.01, 0.08]),
                   nusigmaf=np.array([0, 0.135]),
                   chi=np.array([1, 0]),
                   diffusion=np.array([1.5, 0.4]))
mat2 = Fissionable("Fuel2", scatter=np.array([[0, 0], [0.02, 0]]),
                   absorb=np.array([0.01, 0.085]),
                   nusigmaf=np.array([0, 0.135]),
                   chi=np.array([1, 0]),
                   diffusion=np.array([1.5, 0.4]))
mat3 = Fissionable("Fuel2+Rod", scatter=np.array([[0, 0], [0.02, 0]]),
                   absorb=np.array([0.01, 0.13]),
                   nusigmaf=np.array([0, 0.135]),
                   chi=np.array([1, 0]),
                   diffusion=np.array([1.5, 0.4]))
mat4 = Isotope("Reflector", scatter=np.array([[0, 0], [0.04, 0]]),
               absorb=np.array([0, 0.01]),
               diffusion=np.array([2, 0.3]))
mat5 = Isotope("Reflector+Rod", scatter=np.array([[0, 0], [0.04, 0]]),
               absorb=np.array([0, 0.055]),
               diffusion=np.array([2, 0.3]))

## Defining the geometry

In [3]:
lengths_x = np.array([10, 20, 20, 20, 20, 20, 20, 20, 20])
lengths_z=np.array([20,260,80,20])
incore = [1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 0
    , 1, 1, 1, 1, 1, 1, 1, 1, 0
    , 1, 1, 1, 1, 1, 1, 1, 0, 0
    , 1, 1, 1, 1, 1, 1, 0, 0, 0
    , 1, 1, 1, 1, 0, 0, 0, 0, 0]
black = [i for i in range(81) if incore[i] == 0]
plane = Cartesian([lengths_x, lengths_x],
                  [ Reflector(),Void(),Reflector(), Void()],
                  black)

## Defining the materials composition

In [4]:
composition = [4, 4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4,
               4, 4, 4, 4,
               3, 2, 2, 2, 3, 2, 2, 1, 4,
               2, 2, 2, 2, 2, 2, 2, 1, 4,
               2, 2, 2, 2, 2, 2, 1, 1, 4,
               2, 2, 2, 2, 2, 2, 1, 4, 4,
               3, 2, 2, 2, 3, 1, 1, 4,
               2, 2, 2, 2, 1, 1, 4, 4,
               2, 2, 1, 1, 1, 4, 4,
               1, 1, 1, 4, 4, 4,
               4, 4, 4, 4,
               3, 2, 2, 2, 3, 2, 2, 1, 4,
               2, 2, 2, 2, 2, 2, 2, 1, 4,
               2, 2, 3, 2, 2, 2, 1, 1, 4,
               2, 2, 2, 2, 2, 2, 1, 4, 4,
               3, 2, 2, 2, 3, 1, 1, 4,
               2, 2, 2, 2, 1, 1, 4, 4,
               2, 2, 1, 1, 1, 4, 4,
               1, 1, 1, 4, 4, 4,
               4, 4, 4, 4,
               5, 4, 4, 4, 5, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 5, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4, 4,
               5, 4, 4, 4, 5, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4, 4,
               4, 4, 4, 4, 4, 4,
               4, 4, 4, 4]
names = {1: mat1, 2: mat2, 3: mat3, 4: mat4, 5: mat5}
isotopes = [names[c] for c in composition]

## Refining the geometry
The refinment is done by hand and not using the refinment tools in order to save memory and run time

In [5]:
subcells = 2
split = [np.array([1*subcells]+[2*subcells]*8)]*2
refined_plane = plane.refine_mesh(split).to_polybox()

In [6]:
z_neighbors = [(Void(), 1)]+[ (a, a+2) for a in range(subcells*19-2)]+[(subcells*19-2, Void())]
neighbors = [
    [n if isinstance(n, Boundary) else n + h * refined_plane.cells for n in
     neigh]
    + list(map(lambda n: n if isinstance(n, Boundary)
    else n * refined_plane.cells + cell, pair)) for h, pair in
    enumerate(z_neighbors)
    for cell, neigh in enumerate(refined_plane.neighbors)]
z_lengths = [20/subcells]*19*subcells
lengths = [list(plane_length) + [z_length]
           for z_length in z_lengths for plane_length in refined_plane.lengths]
refined_geo = PolyBox(3, neighbors, lengths)

In [7]:
incore = [1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 1
    , 1, 1, 1, 1, 1, 1, 1, 1, 0
    , 1, 1, 1, 1, 1, 1, 1, 1, 0
    , 1, 1, 1, 1, 1, 1, 1, 0, 0
    , 1, 1, 1, 1, 1, 1, 0, 0, 0
    , 1, 1, 1, 1, 0, 0, 0, 0, 0]*4
black = [i for i in range(81*4) if incore[i] == 0]
original_geometry = Cartesian([lengths_x, lengths_x,lengths_z],
                  [Reflector(),Void(), Reflector(),Void(),
                  Void(),Void()],
                  black)

## Defining the core and the refined core

In [8]:
system = Core(isotopes,2,original_geometry, CurrentCalculatorFD.from_isotopes(isotopes))

In [9]:
spliting = [[subcells]+[2*subcells]*8]*2+[[subcells,13*subcells,4*subcells,subcells]]
spliting

[[2, 4, 4, 4, 4, 4, 4, 4, 4], [2, 4, 4, 4, 4, 4, 4, 4, 4], [2, 26, 8, 2]]

In [10]:
calc = system.current_calc.mesh_refine(system.geometry, spliting, refined_geo)
assemblies = system.geometry.array_refine(np.arange(system.geometry.cells), spliting).astype(int)
isotopes = [system.isotopes[assembly] for assembly in assemblies]
core = Core(isotopes,2,refined_geo,calc)

## Solving the diffusion equation, reference eigenvalue is 1.02909

In [11]:
k,flux=solve_k_diffusion(core,rtol_vector = 1e-8,
                      lin_rtol=1e-10)

In [12]:
assert abs(k-1.02909)<1e-3

## Computing the fission_source

In [13]:
thermal = flux[1::2]
fission_source = thermal*np.array([iso.nusigmaf[1] for iso in core.isotopes])
fuel = [i for i,iso in enumerate(core.isotopes) if iso.isfissile]
core_source = fission_source[fuel]

## Computing the ppf

In [14]:
np.max(core_source)/np.mean(core_source)

2.515861766100126

## Computing the radial power distribution

In [15]:
intermidate_split = [np.array([1]+[1]*8)]*2+[np.array([1,13,4,1])]
intermidte_geo = original_geometry.refine_mesh(intermidate_split)
rest_split = [[subcells]+[2*subcells]*8,[subcells]+[2*subcells]*8,[subcells]*19]
fission_source = np.bincount(intermidte_geo.array_refine(np.arange(intermidte_geo.cells), rest_split).astype(int),fission_source)
rsource = np.reshape(fission_source,(len(fission_source)//(19),19),order="F")

In [16]:
radial = np.mean(rsource,axis=1)
radial = radial/[intermidte_geo.volumes[i] for i in range(69)]
radial = radial[radial.nonzero()]
radial/np.mean(radial)

array([0.75025776, 1.37840092, 1.50805463, 1.2650098 , 0.60462858,
       0.95923199, 0.92577075, 0.69703655, 1.37840092, 1.49068617,
       1.5157602 , 1.35481486, 1.10924622, 1.05345998, 0.93789055,
       0.67691523, 1.50805463, 1.5157602 , 1.43653406, 1.36444336,
       1.20419603, 1.07690043, 0.9544119 , 0.60669351, 1.2650098 ,
       1.35481486, 1.36444336, 1.21626793, 0.9902628 , 0.90244783,
       0.78436721, 0.60462858, 1.10924622, 1.20419603, 0.9902628 ,
       0.45259533, 0.67484598, 0.52341877, 0.95923199, 1.05345998,
       1.07690043, 0.90244783, 0.67484598, 0.51724262, 0.92577075,
       0.93789055, 0.9544119 , 0.78436721, 0.52341877, 0.69703655,
       0.67691523, 0.60669351])

## Computing the axial power distribution

In [17]:
axial = np.mean(rsource,axis=0)
17.026 * axial/np.sum(axial)

array([0.        , 0.30385148, 0.58272255, 0.84975262, 1.08593671,
       1.28269008, 1.43289057, 1.53112867, 1.57391751, 1.55984284,
       1.48965411, 1.36630643, 1.19497917, 0.98302179, 0.74255686,
       0.53261082, 0.34295886, 0.17117892, 0.        ])

## Compare NEM with finite differences

In [18]:
subcells=1
spliting = [[subcells]+[2*subcells]*8]*2+[[subcells,13*subcells,4*subcells,subcells]]
spliting

[[1, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 2, 2, 2, 2], [1, 13, 4, 1]]

In [19]:
from dorban.nem.solve_nem import NEMSettings
from dorban.nem.cmfd_current_calculator import CMFDCurrentCalculator
from dorban.system import mesh_refinement
from dorban.solve_equation import solve_k

nem_system=Core(system.isotopes,system.E,system.geometry,CMFDCurrentCalculator(system.current_calc.dc,3))
nem_system=mesh_refinement(nem_system,spliting)
nem_k,nem_flux=solve_k(nem_system,NEMSettings(split=None))

In [20]:
assert abs(nem_k-k)<1e-3

## compare axial power distribution

In [21]:
nem_flux=nem_flux/np.sum(nem_flux)

In [22]:
nem_thermal = nem_flux[1::2]
nem_fission_source = nem_thermal*np.array([iso.nusigmaf[1] for iso in nem_system.isotopes])
fuel = [i for i,iso in enumerate(nem_system.isotopes) if iso.isfissile]
nem_core_source = nem_fission_source[fuel]

In [23]:
intermidate_split = spliting
intermidte_geo = original_geometry.refine_mesh(intermidate_split)
rest_split = intermidte_geo.uniform_split(1)
nem_fission_source = np.bincount(intermidte_geo.array_refine(np.arange(intermidte_geo.cells), rest_split).astype(int),nem_fission_source)
nem_rsource = np.reshape(nem_fission_source,(len(nem_fission_source)//(19),19),order="F")

In [24]:
nem_axial = np.mean(nem_rsource,axis=0)
17.026 * nem_axial/np.sum(nem_axial)

array([0.        , 0.3467046 , 0.59815473, 0.85882832, 1.08780944,
       1.27806584, 1.42281154, 1.51694028, 1.55718168, 1.54224195,
       1.47288761, 1.35198236, 1.18451225, 0.97726047, 0.74515898,
       0.53928469, 0.35242384, 0.19375141, 0.        ])

In [25]:
nem_normalized_axial=(nem_axial/np.sum(nem_axial))[1:-1]
normalized_axial=(axial/np.sum(axial))[1:-1]
assert np.max(np.abs((nem_normalized_axial-normalized_axial)/normalized_axial))<0.2